# Lexical of 20th century Odyssey translations (Part B): Etymologies


____________________________________________________
## **Road Map**

**I. Libraries, files, and paths**

**II. The Texts**

1. Bibliographic information about the translators  
2. The translators at a glance tokenwise   

**III. TTR Analysis**

1. All-in, straightforward model  
    a) TTR Computation  
    b) Shapiro-Wilk test to check for normality  
    c) One-wat ANOVA for overall differences  
    d) Pairwise t-test using Bonferroni coprrection  
    c) Meassuring effect size ussing Cohen's d  

2. Adaptive models  
    a) Mixed-Effects model: author fixed effect / book as random effect  
    b) Standardized TTR:   
    c) Moving-average TTR: translation as temporal change  

3. Supplement models  
    a) Lexical Density   
    b) Diachronic analysis  
    c) Semantic fields:  

**IV. Zipf's Law**

**V. TF-IDF**



**VI. Discussing Results**

In [6]:
import autotime # Provision for anxious people
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 1.38 ms (started: 2025-04-20 23:49:08 +02:00)


In [8]:
# ----------------------------------------------------------------------
# Baic Libraries
# ----------------------------------------------------------------------

import sys 
import os

import ast
from collections import Counter

import re
import nltk

import numpy as np
import pandas as pd

import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm

import scipy.stats as stats
from itertools import combinations

In [9]:
# ----------------------------------------------------------------------
# Personalized Visualization & Functions
# ---------------------------------------------------------------------- 

sys.path.append('/Users/debr/English-Homer/functions') 
import matplotlib.pyplot as plt
import seaborn as sns

import e_chroma as chroma # My Vizualization library
import e_plots as oz      # My custom plots library
import e_pandisplay as pan# My pandas display options

import e_nlp_ody as e     # Import my nlp functions

import warnings           # Nononsense provision
warnings.filterwarnings('ignore')


* Got some chroma in your soma, Oma!
	 »----> use chroma.save_figure(fig, 'my_plot')
Default output path: ./Homer_xplots/

* OZ is behind the curtain!
	 »----> use oz.<func>
	 »----> also, oz goes chroma (for styling)!

*Has Pan taken over?
✓ Pandas display set to e_pandisplay defaults!
	 »----> use pan.<func>

* The editor is in the house!
	 »----> use e.<func> e.g. nlp = e.NLPPipeline(language='english')

Stopwords customized:
  Added: {'were', 'ten', 'three', 'been', 'upon', 'two', 'seven', 'four', 'n', 'there', 'mrs', 'said', 'nine', 'this', 'eight', 'of', 'it', 'six', 'mr', 'they', 'he', 'that', "'", 'she', 'one', 'them', 'was', 'be', 'is', 'five', 'being', "'and", 'are'}
  Removed: {''}
  Total stopwords: 215
Punctuation customized:
  Keeping: {'', '-'}
  Additional removals: {'—', '-', '…', '\\', "'", ',\n        "\'",\n        '}
  Punctuation to be removed: !"#$%&'()*+,,
        "'",
        -./:;<=>?@[\]^_`{|}~—…


[nltk_data] Downloading package punkt_tab to /Users/debr/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
# ----------------------------------------------------------------------
# File management
# ----------------------------------------------------------------------

# TO UPDATE
nb_id = "lexical_B01"

output_path = f"./"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
output_path_plots = f"./{output_path}/{nb_id}_plots/"
os.makedirs(os.path.dirname(output_path_plots), exist_ok=True)
chroma.set_output_path(output_path_plots)

Output path set to: ././/lexical_B01_plots/


In [11]:
# ----------------------------------------------------------------------
# Odysseys
# ----------------------------------------------------------------------

translators = ['AT_Murray', 'Fitzgerald', 'Lattimore', 'Fagles', 'Wilson', 'Green', 'Woolf']

dfs = []

for odyssey in translators:
    filepath = f"/Users/debr/odysseys_en/dataframed/Odyssey_{odyssey}_DataFrame.csv"
    temp_df = pd.read_csv(filepath)  
    dfs.append(temp_df)  # Append it to the list

df = pd.concat(dfs, axis=0, ignore_index=True)

df["text"] = df["text"].apply(ast.literal_eval)
df["tokens"] = df["tokens"].apply(ast.literal_eval)
df['translator'] = pd.Categorical(df['author'])
df["book_num"] = pd.Categorical(df["book_num"])
df = df[['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens']]

# ----------------------------------------------------------------------
# Backup dataframe only 'translator', 'book_num', 'text', 'tokens', columns
# ----------------------------------------------------------------------
df_bkp = df[['translator', 'book_num', 'text', 'tokens']].copy()
# ----------------------------------------------------------------------
# Dataframe check
# ----------------------------------------------------------------------

e.check_df(df)

Mr righteous here has no missing values!

* df columns: Index(['translator', 'book_num', 'text', 'tokens', 'num_words', 'num_tokens'], dtype='object') 

* Shape: (168, 6) 

* Total memory in MB: 4.064749


## **2. The Dictionaries

Python etymologies package is Ety. It is based on Melo's dictionary. However, it doesn't works as expected. So I had to came up with my own solution.

This is the data-dictionary. 

Etymological Wordnet 2013-02-08
Gerard de Melo
http://icsi.berkeley.edu/~demelo/etymwn/


== DESCRIPTION ==

The Etymological Wordnet project provides information about how words in different languages 
are etymologically related. The information is mostly mined from the English version of
Wiktionary, but also contains a number of manual additions.


== FORMAT ==

The package includes a Tab-separated values (TSV) file in UTF-8 format with three columns,
providing a word, a relation, and a target word. Words are given with ISO 639-3 codes
(additionally, there are some ISO 639-2 codes prefixed with "p_" to indicate proto-languages).
The most relevant relation is "rel:etymology". To see only etymological relations, run
  grep "rel:etymology" etymwn.tsv | less
on UNIX-based systems.


== CREDITS AND LICENSE ==

Gerard de Melo
http://icsi.berkeley.edy/~demelo/
Based on the contributions of the English Wiktionary community
http://en.wiktionary.org/

License: CC-BY-SA 3.0

In scientific works, please cite:
  Gerard de Melo, Gerhard Weikum. "Towards Universal Multilingual Knowledge Bases".
  In: Principles, Construction, and Applications of Multilingual Wordnets. Proceedings
  of the 5th Global Wordnet Conference (GWC 2010). Narosa Publishing 2010, New Delhi India.

### Working with ety

In [ ]:
# ----------------------------------------------------------------------
# Probing ety
# ----------------------------------------------------------------------
import ety
# Get the etymology of the word "muse" in English
word = "muse"
ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

Etymology of 'muse': [Word(muse, Middle French (ca. 1400-1600) [frm])]
Etymology tree of 'muse': muse (English)
└── muse (Middle French (ca. 1400-1600))
    └── Musa (Latin)
        └── Μοῦσα (Ancient Greek (to 1453))


Curiously, ety doesn't have the full etymology for "muse".

In [6]:
# Get the etymology of the word "table" in English

word = "table"
ety_origin = ety.origins(word)
ety_tree = ety.tree(word)
print(f"Etymology of '{word}': {ety_origin}")
print(f"Etymology tree of '{word}': {ety_tree}")

Etymology of 'table': [Word(table, Middle English (1100-1500) [enm])]
Etymology tree of 'table': table (English)
└── table (Middle English (1100-1500))


### Developing a dictionary we can use


In [ ]:
# ----------------------------------------------------------------------
# Getting the full dictionary
# ----------------------------------------------------------------------

etymologypath = "/Users/debr/odysseys_en/etymwn-20130208/etymwn.tsv"

etymology_df = pd.read_csv(etymologypath, sep="\t", names=["word", "relation", "target_word"], encoding="utf-8")
etymology_df["relation"] = etymology_df["relation"].astype("category") # to save space

e.check_df(etymology_df)

Mr righteous here has no missing values!

* df columns: Index(['word', 'relation', 'target_word'], dtype='object') 

* Shape: (6031431, 3) 

* Total memory in MB: 922.924706


In [13]:
etymology_df.sample(10, random_state=402)

,word,relation,target_word
5463534,spa: condicional,rel:has_derived_form,spa: condicionales
467142,eng: albuminization,rel:etymologically_related,eng: toxalbumin
2075742,fra: insulteur,rel:etymologically_related,fra: insulteuse
4897623,lat: torculo,rel:has_derived_form,lat: torculavissent
255143,deu: anfangen,rel:has_derived_form,deu: anfingt
4675182,lat: prodicimini,rel:is_derived_from,lat: prodico
1992445,fra: enflai,rel:is_derived_from,fra: enfler
5773738,spa: sobrecargasen,rel:is_derived_from,spa: sobrecargar
4882662,lat: taedeo,rel:has_derived_form,lat: taedueritis
5357602,spa: aclaraseis,rel:is_derived_from,spa: aclarar


In [14]:
relations = [etymology_df['relation'].unique()]
relations

[['rel:etymological_origin_of', 'rel:has_derived_form', 'rel:is_derived_from', 'rel:etymology', 'rel:etymologically_related', 'rel:variant:orthography', 'rel:derived', 'rel:etymologically']
 Categories (8, object): ['rel:derived', 'rel:etymological_origin_of', 'rel:etymologically', 'rel:etymologically_related', 'rel:etymology', 'rel:has_derived_form', 'rel:is_derived_from', 'rel:variant:orthography']]

**Description of categories**

| Relation                        | Description |
|----------------------------------|------------|
| **rel:etymological_origin_of**   | Indicates that a word is the root or ancestor of another word. Example: Latin *scientia* is the etymological origin of English *science*. |
| **rel:has_derived_form**         | Shows that a word has a derived variant. Example: *happy* has the derived form *happiness*. |
| **rel:is_derived_from**          | Specifies that a word originates from another word. This is the inverse of *rel:etymological_origin_of*. Example: English *science* is derived from Latin *scientia*. |
| **rel:etymology**                | A general relation indicating the etymology of a word without specifying a direction of derivation. |
| **rel:etymologically_related**   | Indicates that two words share a common etymology but are not directly derived from one another. Example: English *hospital* and *host* are etymologically related through Latin *hospes*. |
| **rel:variant:orthography**      | Refers to different spellings of the same word. Example: *color* (American English) vs. *colour* (British English). |
| **rel:derived**                  | A broad category that includes words that evolved from another language but does not specify the exact relationship. |
| **rel:etymologically**           | A general tag used to indicate some form of etymological connection. Often used when the exact type of relation is unclear. |

In [15]:
target_word = "enm: table"  # Change this to any word you want to match

etymology_root_1 = etymology_df[etymology_df["word"] == target_word]

print(etymology_root_1)

        word        relation                    target_word
1345511  enm: table  rel:etymological_origin_of  eng: table


In [16]:
# ----------------------------------------------------------------------
# A subset dictionary based on "relation"
# ----------------------------------------------------------------------

relation_etymology_df = etymology_df[etymology_df["relation"] == "rel:etymology"]
relation_etymology_df.sample(8, random_state=42)

,word,relation,target_word
1727960,fin: räkänokka,rel:etymology,fin: nokka
753310,eng: futureless,rel:etymology,eng: future
842271,eng: inveracity,rel:etymology,eng: in-
804350,eng: homœostasis,rel:etymology,eng: homœ-
1622981,fin: huumekauppias,rel:etymology,fin: huume
404060,eng: Ceylon,rel:etymology,grc: Σελεδίβα
1063714,eng: pritumumab,rel:etymology,eng: -tum-
5429877,spa: cabezota,rel:etymology,spa: -ota


In [17]:
# ----------------------------------------------------------------------
# Chasing the etymology tree
# ----------------------------------------------------------------------

target_word = "eng: ghost"
etymology_root_1 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("Root 1:", etymology_root_1)

target_word = "enm: gost" 
etymology_root_2 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("\nRoot 2:", etymology_root_2)

target_word = "ang: gast" 
etymology_root_3 = relation_etymology_df[relation_etymology_df["word"] == target_word]
print("\nRoot 3:", etymology_root_3)

Root 1:        word        relation       target_word
761941  eng: ghost  rel:etymology  enm: gost 

Root 2:         word       relation       target_word
1342198  enm: gost  rel:etymology  ang: gast 

Root 3: Empty DataFrame
Columns: [word, relation, target_word]
Index: []


In [18]:
# ----------------------------------------------------------------------
# Function to get the etymology tree
# ----------------------------------------------------------------------

def trace_etymology(word, df):
    etymology_chain = [word]  # Store the lineage of words
    
    while True:
        # Find the row where 'word' matches
        etymology_row = df[df["word"] == word]
        
        # If no match is found, stop the loop
        if etymology_row.empty:
            break
        
        # Get the next word in the etymological chain
        next_word = etymology_row["target_word"].values[0]
        
        # Append to the chain and set up for the next iteration
        etymology_chain.append(next_word)
        word = next_word  # Set the next search target

    # Format output sentence
    if len(etymology_chain) > 1:
        etymology_str = ' → '.join(etymology_chain)
        print(f'The word "{etymology_chain[0]}" traces back through: {etymology_str}.')
    else:
        print(f'No etymological root found for "{word}".')

# Example usage
trace_etymology("eng: ghost", relation_etymology_df)

The word "eng: ghost" traces back through: eng: ghost → enm: gost → ang: gast.


In [20]:
# ----------------------------------------------------------------------
# The function in a loop for a list of words
# ----------------------------------------------------------------------

ety_inquiries = ['muse', 'ghost', 'table', 'sword', 'shield', 'spear', 'battle', 'warrior', 'hero', 'goddess']

for word in ety_inquiries:
    print(f"Word: {word}")
    trace_etymology(f"eng: {word}", relation_etymology_df)
    print("\n")

Word: muse
The word "eng: muse" traces back through: eng: muse → frm: muse → lat: Musa → grc: Μοῦσα.


Word: ghost
The word "eng: ghost" traces back through: eng: ghost → enm: gost → ang: gast.


Word: table
The word "eng: table" traces back through: eng: table → enm: table.


Word: sword
The word "eng: sword" traces back through: eng: sword → enm: sword.


Word: shield
The word "eng: shield" traces back through: eng: shield → ang: scieldan.


Word: spear
The word "eng: spear" traces back through: eng: spear → ang: spere.


Word: battle
The word "eng: battle" traces back through: eng: battle → enm: batel → fro: bataille.


Word: warrior
No etymological root found for "eng: warrior".


Word: hero
No etymological root found for "eng: hero".


Word: goddess
The word "eng: goddess" traces back through: eng: goddess → eng: -ess → fra: -esse → lat: -issa → grc: -ισσα.


